In [ ]:
from pathlib import Path
import chromadb


from rag_build.loading import load_vault
from rag_build.chunking import chunk_all_documents, Chunk
from rag_build.embedding import embed_texts
from rag_build.vector_store import index_chunks, get_collection


VAULT_DIR = Path.cwd().parent / 'data' / '01-Regression'
VAULT_DIR

WindowsPath('c:/Users/eoinm/Documents/rag-build-project/data/01-Regression')

In [2]:

vault = load_vault(VAULT_DIR)
chunks = chunk_all_documents(vault)

index_chunks(chunks)

chunks[0]


Chunk(text='# LinearRegression > Linear Regression\n\nLinear regression predicts a continuous outcome by fitting a straight line, or a flat plane in higher dimensions, through the data. It is the simplest and most interpretable regression model and a baseline against which more complex models are compared.\n', metadata={'source': 'LinearRegression.md', 'file': 'LinearRegression', 'headings': ['Linear Regression'], 'index': 0, 'section': '01-Regression', 'topic': 'Regression', 'ml_task': 'supervised', 'related_topics': ['PolynomialRegression', 'SupportVectorRegression', 'LinearAlgebra'], 'summary': 'A supervised method that fits a straight-line relationship between input features and a continuous target value.', 'keywords': ['regression', 'least squares', 'coefficients', 'gradient descent', 'prediction', 'linear model']})

In [4]:


query = 'What is a Support Vector Machine?'

collection = get_collection()

query_vector = embed_texts([query])[0]

results = collection.query(query_embeddings=[query_vector],n_results=3)

top_matches = []
for text, metadata, distance in zip(results['documents'][0],results['metadatas'][0],results['distances'][0]):
    top_matches.append({
        'text':text,
        'metadata':metadata,
        'distance':distance
    })

top_matches

[{'text': '# SupportVectorRegression > Support Vector Regression\n\nSupport vector regression (SVR) adapts the support vector machine framework to predict continuous values. Instead of fitting a line that minimises every error, it fits a band of tolerance and only penalises predictions that fall outside it.\n',
  'metadata': {'ml_task': 'supervised',
   'summary': 'A regression method that fits a margin of tolerance around the data and uses kernels to model non-linear relationships.',
   'related_topics': 'SupportVectorMachines,LinearRegression,PolynomialRegression',
   'section': '01-Regression',
   'headings': 'Support Vector Regression',
   'keywords': 'svr,epsilon margin,kernel,support vectors,regression,robust',
   'index': 0,
   'topic': 'Regression',
   'file': 'SupportVectorRegression',
   'source': 'SupportVectorRegression.md'},
  'distance': 0.4183231592178345},
 {'text': '# 03-SupportVectorRegression > Support Vector Regression\n\nThis method is regression equivalent of clas

In [ ]:
from rag_build.vector_store import search
query = 'Explain Support Vector Machine?'

where = {'related_topics':{"$in":['Linear Regression']}}
hits = search(query=query,top_k=3)
hits[0]


{'text': '# SupportVectorRegression > Support Vector Regression\n\nSupport vector regression (SVR) adapts the support vector machine framework to predict continuous values. Instead of fitting a line that minimises every error, it fits a band of tolerance and only penalises predictions that fall outside it.\n',
 'metadata': {'keywords': 'svr,epsilon margin,kernel,support vectors,regression,robust',
  'headings': 'Support Vector Regression',
  'summary': 'A regression method that fits a margin of tolerance around the data and uses kernels to model non-linear relationships.',
  'ml_task': 'supervised',
  'file': 'SupportVectorRegression',
  'section': '01-Regression',
  'index': 0,
  'topic': 'Regression',
  'source': 'SupportVectorRegression.md',
  'related_topics': 'SupportVectorMachines,LinearRegression,PolynomialRegression'},
 'distance': 0.3993566036224365}

In [6]:

examples = []

for i,hit in enumerate(hits,1):

    index = f'[{i}]'
    file = hit['metadata']['file']
    headings = hit['metadata']['headings'].split(',')
    breadcrumb = f'<{index} {file}: {' > '.join(headings)}>'
    full_text = f'{breadcrumb}\n\n{hit['text']}' 
    print(full_text)




<[1] SupportVectorRegression: Support Vector Regression>

# SupportVectorRegression > Support Vector Regression

Support vector regression (SVR) adapts the support vector machine framework to predict continuous values. Instead of fitting a line that minimises every error, it fits a band of tolerance and only penalises predictions that fall outside it.

<[2] 03-SupportVectorRegression: Support Vector Regression > Basic Principle>

# 03-SupportVectorRegression > Support Vector Regression > Basic Principle

Much like Support Vector Machines in general, the idea of SVR, is to find a plane(linear or otherwise) in a plane that allows us to make accurate predictions for future data. The regression is done in a way that all the currently available datapoints fit in an error width given by ![epsilon_small](http://mathurl.com/ybr3ffkc.png). This allows us to find a plane which fits the data best and then this can be used to make future predictions on more data. 

> Sometimes a soft error margin 

In [7]:
chunks = search(query=query,top_k=2)

print(chunks[0])

contains = {'headings':'basic principles'}

# Need to write a function that identifies what used to be a list and convert it back to list
hits = []
for chunk in chunks:
    for key,value in contains.items():
        if value.lower() in chunk['metadata'][key].lower():
            print(f'Index {chunk['metadata']['index']}:{chunk['metadata'][key]} == {value}')
            hits.append(chunk)

hits

{'text': '# SupportVectorRegression > Support Vector Regression\n\nSupport vector regression (SVR) adapts the support vector machine framework to predict continuous values. Instead of fitting a line that minimises every error, it fits a band of tolerance and only penalises predictions that fall outside it.\n', 'metadata': {'index': 0, 'keywords': 'svr,epsilon margin,kernel,support vectors,regression,robust', 'file': 'SupportVectorRegression', 'source': 'SupportVectorRegression.md', 'topic': 'Regression', 'summary': 'A regression method that fits a margin of tolerance around the data and uses kernels to model non-linear relationships.', 'section': '01-Regression', 'headings': 'Support Vector Regression', 'ml_task': 'supervised', 'related_topics': 'SupportVectorMachines,LinearRegression,PolynomialRegression'}, 'distance': 0.3993566036224365}


[]

In [ ]:
# Filtering by Distance Threshold
threshold = 0.41
similar = []

for chunk in chunks:
    if chunk['distance'] <= threshold:
        similar.append(chunk)

similar


[{'text': '# 03-SupportVectorRegression > Support Vector Regression\n\nThis method is regression equivalent of classification using [Support Vector Machines](../02-Classification/03-SupportVectorMachines.md).\n',
  'metadata': {'file': '03-SupportVectorRegression',
   'related_topics': 'Support Vector Machines,Linear Regression,Polynomial Regression',
   'section': 'Regression',
   'headings': 'Support Vector Regression',
   'ml_task': 'supervised',
   'index': 0,
   'topic': 'Support Vector Regression',
   'keywords': 'SVR,support vector regression,epsilon,kernel,RBF,penalty parameter,feature scaling,nonlinear regression,soft margin',
   'summary': "Explains SVR's core principle of fitting data within an epsilon error margin. Covers nonlinearity via preprocessing and kernel tricks (RBF), the soft error margin zeta, and the penalty parameter C for controlling overfitting. Notes that feature scaling must be done manually in Python.",
   'source': '03-SupportVectorRegression.md'},
  'dis